# Phase 2 - State tracking on assembly_2

This notebook builds the state of every part of the figure over time, starting from the labels of Phase 1. It goes from patches to objects, then from objects to events.

## Section 1 - Purpose and questions

Phase 1 labels every patch of every frame. It does not know which patches belong to the same part, and it does not remember anything from one frame to the next.

The positions study showed why this matters. The relative positions held while the regions were the right parts. Every false alarm looked at came from a wrong region: the controller of the headset labelled as body, objects on the desk, fragments of parts hidden by the hands.

So the first step of Phase 2 is to turn patches into objects. Each part of the figure becomes a track, and a track keeps its identity over time: a part does not change what it is, only where it is.

The number of tracks comes from the dependency graph, one for each copy of a part: 11 tracks, with two ears. No new identity is ever created. This differs from SORT, where an object that reappears gets a new identity. Here the inventory is known, so a part found again is the same part.

A region is given to a track only if it has the class of the part and appears near the region the track had in the previous frame. A region that fits no track is discarded. This follows the rule of TubeletGraph, where a new region is kept only if it is both near the object and similar to it; here the similarity is the class given by Phase 1. It should discard the controller.

Between two consecutive frames the head moves little, so nearness in the image is enough. The positions study asks for more: a region accepted where the part can be with respect to the figure, because the head moves. This matters when a part is found again after a long time. For a mounted part it is left to the state of the parts, where the part has a parent to be measured against. For a part found again on the table it remains an open problem.

A part that is not seen is not visible, but it stays where it was. The state changes only with a positive proof, never because something is missing.

The rules need a few thresholds: how near a region must be, how small a region can be, for how many frames a part must be seen before its track starts. For each threshold the data of assembly_2 give an allowed interval. The value inside the interval is chosen later in this notebook, as the one that gives the fewest wrong events, with the centre of the interval in case of a tie, following the evaluation plan of Phase 2.

Three questions are asked. How many regions does each class have in a frame, compared with the copies in the graph? How far does a part move between two frames, on the table, in hand and mounted? Can each track follow its part through the whole video, discarding the foreign objects?

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

phase_2_positions_study.ipynb, Sections 4.5 and 6: the relative positions hold when the regions are the right parts; every false alarm comes from a wrong region.

Bewley et al., Simple Online and Realtime Tracking, ICIP 2016, Section 3.4: a new track undergoes a probationary period before it is accepted; an object that reappears gets a new identity.

Sun et al., Tracking and Understanding Object Transformations (TubeletGraph), NeurIPS 2025, Section 3.3 and Figure 2: a new region is kept only if it satisfies both proximity and semantic consistency with the object.

configs/dependency_graph.json: the parts and their number of copies.

Evaluation plan of Phase 2: thresholds chosen in an interval given by the data, as the value with the fewest wrong events.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

All paths and settings in one place. C-RADIO is not loaded: Phase 2 works on the labels of Phase 1, already saved for every frame of assembly_2.

The labels are those of the support set A ("current"). The positions study recomputed 10 frames with the final code and found them identical, so they are used here as they are.

The parts and their number of copies are read from the dependency graph. Every part name must be a class of Phase 1, otherwise its track could never see it.

The thresholds of the tracking are added to the settings once their interval is measured, in the sections that follow. Until the events are available they take the centre of their interval, as a provisional value; the final value is chosen with the events.

In [1]:
# 2.1 Imports, paths and shared module
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

from perception.classes import CLASS_NAMES, PIECE_LABELS

VIDEO_PATH = PROJECT_ROOT / "data" / "raw_videos" / "assembly_2.mp4"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"   # annotated frames (labelme json), used to check identities
EVENTS_PATH = PROJECT_ROOT / "configs" / "assembly_2_events.json"
GRAPH_PATH = PROJECT_ROOT / "configs" / "dependency_graph.json"
LABELS_PATH = PROJECT_ROOT / "outputs" / "phase_1_support_set_v2_study" / "assembly_2_labels_current_protected.npz"  # read only
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_2_state_tracking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"VIDEO_PATH": VIDEO_PATH, "FRAMES_DIR": FRAMES_DIR, "EVENTS_PATH": EVENTS_PATH,
                   "GRAPH_PATH": GRAPH_PATH, "LABELS_PATH": LABELS_PATH}.items():
    print(f"[{'ok' if path.exists() else 'MISSING'}] {name}: {path.relative_to(PROJECT_ROOT)}")

[ok] VIDEO_PATH: data\raw_videos\assembly_2.mp4
[ok] FRAMES_DIR: data\extracted_frames
[ok] EVENTS_PATH: configs\assembly_2_events.json
[ok] GRAPH_PATH: configs\dependency_graph.json
[ok] LABELS_PATH: outputs\phase_1_support_set_v2_study\assembly_2_labels_current_protected.npz


In [2]:
# 2.2 Settings
# Values of the pipeline (phase_1_perception.ipynb, Section 2.5).
INPUT_SIZE, PATCH_SIZE = (1024, 1024), 16
GRID = (INPUT_SIZE[0] // PATCH_SIZE, INPUT_SIZE[1] // PATCH_SIZE)   # 64 x 64 patches
# Thresholds of the tracking: added here once their interval is measured.

In [ ]:
# 2.3 Labels of assembly_2, event log and dependency graph
stored = np.load(LABELS_PATH)
LABELS = stored["current"]                     # support set A, checked in phase_2_positions_study, Section 3
assert [str(c) for c in stored["class_names"]] == CLASS_NAMES, "Class order of the saved labels must match CLASS_NAMES"

cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS, N_FRAMES = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
assert LABELS.shape == (N_FRAMES, *GRID), f"Saved labels {LABELS.shape} do not match the video"

with open(EVENTS_PATH) as f:
    LOG = json.load(f)
with open(GRAPH_PATH) as f:
    GRAPH = json.load(f)
EVENTS = LOG["events"]
PARTS = GRAPH["parts"]
missing = set(PARTS) - PIECE_LABELS
assert not missing, f"Parts of the graph with no class in Phase 1: {missing}"
COPIES = {p: PARTS[p]["count"] for p in PARTS}  # number of tracks for each part

# The 32 validation frames of assembly_2 (phase_1_support_set_v2_study); other annotated frames are not used
VALIDATION_FRAMES = [f"assembly_2_f{n:04d}" for n in (
    3, 4, 5, 8, 10, 12, 17, 18, 19, 24, 26, 28, 30, 33, 37, 42,
    43, 46, 50, 51, 52, 53, 54, 57, 59, 60, 63, 65, 68, 70, 73, 77)]
missing_json = [f for f in VALIDATION_FRAMES if not (FRAMES_DIR / f"{f}.json").exists()]
assert not missing_json, f"Missing annotations: {missing_json}"
not_used = sorted({p.stem for p in FRAMES_DIR.glob("assembly_2_f*.json")} - set(VALIDATION_FRAMES))

print(f"Labels: {LABELS.shape[0]} frames, grid {GRID}; video: {N_FRAMES} frames at {FPS:.3f} fps")
print(f"Graph: {len(PARTS)} parts, {sum(COPIES.values())} copies: {COPIES}")
print(f"Event log: {len(EVENTS)} events, {len(LOG['extra_events'])} extra events, {len(LOG['negatives'])} negatives")
print(f"Validation frames of assembly_2: {len(VALIDATION_FRAMES)}; annotated but not used: {not_used}")

Labels: 2444 frames, grid (64, 64); video: 2444 frames at 29.996 fps
Graph: 10 parts, 11 copies: {'base': 1, 'body': 1, 'eyes': 1, 'nose': 1, 'moustache': 1, 'mouth': 1, 'ear': 2, 'arm_1': 1, 'arm_2': 1, 'hat': 1}
Event log: 11 events, 3 extra events, 3 negatives
Annotated frames of assembly_2: 33
